# M4 (Bonificación +5): Segmentación y Conteo de Pasajeros Visibles

Entrena un modelo de segmentación de instancias (`YOLO11-seg`) para detectar y contar personas visibles dentro del vehículo.

**Regla de privacidad obligatoria (Sección 10):**
- NO se realiza reconocimiento facial.
- NO se guardan recortes de rostros.
- En las imágenes de evidencia se dibuja únicamente la máscara opaca/translúcida y el conteo numérico.

In [ ]:
!pip install -q ultralytics==8.3.49 roboflow==1.1.49
import os, json, torch
from roboflow import Roboflow
from ultralytics import YOLO
from google.colab import drive, userdata

drive.mount('/content/drive')
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
WORKSPACE = 'tu-workspace-roboflow'
PROJECT_NAME = 'pasajeros-vehiculo-seg'
VERSION_NUM = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT_NAME).version(VERSION_NUM).download('yolov11')

In [ ]:
model = YOLO('yolo11n-seg.pt')
model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=70,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project='/content/drive/MyDrive/proyecto_peaje/runs_m4',
    name='m4_pasajeros_seg',
    exist_ok=True
)

best_weights = '/content/drive/MyDrive/proyecto_peaje/runs_m4/m4_pasajeros_seg/weights/best.pt'
metrics = YOLO(best_weights).val(data=f'{dataset.location}/data.yaml')

report = {
    'modelo': 'M4_pasajeros_seg',
    'mask_mAP50': round(float(metrics.seg.map50), 4),
    'mask_mAP50_95': round(float(metrics.seg.map), 4),
    'precision': round(float(metrics.seg.mp), 4),
    'recall': round(float(metrics.seg.mr), 4)
}
print(json.dumps(report, indent=2))

os.makedirs('/content/drive/MyDrive/proyecto_peaje/models', exist_ok=True)
!cp /content/drive/MyDrive/proyecto_peaje/runs_m4/m4_pasajeros_seg/weights/best.pt /content/drive/MyDrive/proyecto_peaje/models/m4_pasajeros_seg.pt
with open('/content/drive/MyDrive/proyecto_peaje/metricas_M4.json', 'w') as f:
    json.dump(report, f, indent=2)